# Assistente de Pneumologia com RAG
**Trabalho Final · Disciplina de Aprendizado de Máquina Supervisionado (UNESP)**

Autor: **Nycolas Mariotto** · Doutorando em Ciências Biomoleculares e Farmacológicas (UNESP/Botucatu)

Um assistente que responde perguntas clínicas sobre **DPOC, fibrose pulmonar e
paracoccidioidomicose (PCM)** baseado em **fontes confiáveis e citáveis**, usando
*Retrieval-Augmented Generation* (RAG).

**Objetivo:** medir se recuperar fontes (e reordená-las com *reranking*) torna as respostas mais
corretas e rastreáveis do que o LLM respondendo sozinho.

> **Motivação:** a ideia nasceu da minha participação no **doutorado da Dra. Erika Mayumi Watanabe**
> (FMB/HC-FMB, orientação do **Prof. Ricardo de Souza Cavalcante**, defendido em **09/06/2026**) sobre as
> **sequelas pulmonares da PCM**, no qual fiz a **análise quantitativa das tomografias**. No pulmão, DPOC,
> enfisema, fibrose e PCM deixam marcas parecidas e se confundem; este assistente cobre DPOC, fibrose e PCM.

Comparei **3 configurações**:
1. **Baseline**: LLM sem recuperação (geração direta, paramétrica).
2. **RAG convencional**: recuperação densa (*embeddings* + FAISS) → LLM.
3. **Proposta**: recuperação densa + **reranking por cross-encoder** que reescora os candidatos e leva o
   trecho mais pertinente ao topo do contexto → LLM.

> **Roda 100% localmente** com um **modelo aberto** (`Qwen2.5-3B-Instruct`, Apache-2.0), na **GPU gratuita
> do Colab (T4)** ou numa GPU local. O modelo é configurável: além do 3B padrão, rodei também o **7B**
> (ablação de tamanho de modelo, §5b). **Todo o código está embutido neste notebook** — nada fica escondido.
>
> Base de conhecimento: **33 documentos abertos (~670 páginas)**, PT/EN, recortados em **2.374** *chunks*.
> Avaliação em **54 perguntas** PT-BR com gabarito rastreável à fonte.

## Como executar
Funciona em **Google Colab** (grátis) ou **JupyterLab local** — a 1ª célula cuida de tudo.
1. (Colab) **Runtime → Change runtime type → T4 GPU**. A geração usa GPU; em CPU fica muito lento.
2. **Run all** (ou célula a célula).

A 1ª célula clona o projeto do **GitHub** (código + base + índice + benchmark + resultados) — ou usa os
arquivos locais, se você já estiver dentro do repositório. O modelo aberto (~6 GB) baixa **uma única vez**
na primeira geração; o índice de busca já vem pronto (não re-processa nada).

In [ ]:
# 1) Carrega o projeto (codigo + base + indice + benchmark + resultados).
#    PORTATIL: se ja estiver dentro do repo (JupyterLab local, mesmo abrindo de notebook/),
#    usa os arquivos LOCAIS; senao (ex.: Colab novo), faz git clone. Idempotente.
import os, subprocess
REPO_URL = "https://github.com/nykemariotto/rag-pneumologia.git"
REPO_DIR = "rag-pneumologia"
def _tem_projeto(d="."):
    return os.path.exists(os.path.join(d, "rag_core.py")) and os.path.isdir(os.path.join(d, "indices"))
if _tem_projeto("."):
    pass                              # ja na raiz do projeto
elif _tem_projeto(".."):
    os.chdir("..")                    # aberto de dentro de notebook/ (JupyterLab local)
else:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)                # Colab / ambiente novo
assert _tem_projeto("."), "Projeto nao encontrado (esperado rag_core.py e a pasta indices/)."
print("Projeto carregado em:", os.getcwd())

In [ ]:
# 2) Instala as dependências (versões fixadas para reprodutibilidade) e confirma o ambiente.
#    O torch com GPU já vem pré-instalado no Colab; aqui instalamos só o restante.
!pip install -q -r requirements.txt
import torch
from importlib.metadata import version
libs = {p: version(p) for p in ["transformers", "sentence-transformers", "faiss-cpu", "accelerate", "rank-bm25"]}
print("Bibliotecas instaladas:", libs)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "(NENHUMA — ative a T4 em Runtime > Change runtime type e rode tudo de novo)")

## 1. Base de conhecimento (o *corpus*)
**33 documentos** abertos e redistribuíveis (~670 páginas), curados por **confiabilidade e licença**:
StatPearls, CDC/MedlinePlus/NHLBI (domínio público), OMS, diretrizes da **SBPT** (J Bras Pneumol),
Consenso Brasileiro de PCM, diretriz **ATS/ERS** de fibrose e revisões *open-access* (PMC/MDPI). Corpus
**bilíngue (PT/EN)**; cada `.txt` retém atribuição e licença (ver `FONTES.md`). Diretrizes com *copyright*
(GOLD, NICE, ATS 2015/2018) entram só como **referência citável**, não no corpus.

In [ ]:
import json
from collections import Counter
meta = json.load(open("indices/meta.json"))
chunks = [json.loads(l) for l in open("indices/chunks.jsonl", encoding="utf-8")]
print(f"Documentos: {meta['n_docs']}  |  Trechos (chunks): {meta['n_chunks']}  |  modelo de embeddings: {meta['emb_model']}")
NOMES = {"dpoc": "DPOC", "fibrose": "Fibrose", "pcm": "PCM"}
cont = Counter(c["disease"] for c in chunks)
print("Trechos por doença:", {NOMES.get(k, k): v for k, v in cont.items()})

### Como o índice é construído (`scripts/indexar.py`)
O código abaixo lê os `.txt`, corta em *chunks*, gera os *embeddings* e monta os índices FAISS (denso) e
BM25 (palavra-chave). Ele **já foi rodado** — o índice pronto vem no repositório, então o notebook **não
re-processa** (é só para o código ficar visível). A célula grava o arquivo, mas não o executa aqui.

In [ ]:
%%writefile scripts/indexar.py
"""
Indexacao da knowledge base (Trabalho Final RAG - Pneumologia).

Le os .txt da knowledge_base/, corta em pedacos ("chunks"), gera os embeddings
(modelo multilingue) e monta os dois indices de busca:
  - FAISS  (busca por significado / densa)  -> indices/faiss.index
  - BM25   (busca por palavra-chave)        -> indices/bm25.pkl
Salva tambem indices/chunks.jsonl (texto + metadados por pedaco) e indices/meta.json.

Roda no .venv do projeto (tem sentence-transformers, faiss, rank-bm25).
O notebook do Colab reusa exatamente esta logica.
"""
import os, re, json, glob, pickle, unicodedata
import numpy as np

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
KB = os.path.join(ROOT, "knowledge_base")
IDX = os.path.join(ROOT, "indices")
EMB_MODEL = os.environ.get("EMB_MODEL", "intfloat/multilingual-e5-base")
CHUNK_SIZE = 1100      # ~caracteres por pedaco
CHUNK_OVERLAP = 150

# ----------------------------- leitura + chunking -----------------------------
def read_doc(path):
    """Le um .txt da KB, separa o cabecalho de atribuicao (metadados) do corpo."""
    with open(path, encoding="utf-8") as f:
        raw = f.read()
    meta = {"title": "", "publisher": "", "url": "", "license": "", "lang": "", "disease": ""}
    body = raw
    if raw.startswith("==="):
        head, _, body = raw.partition("-" * 70)
        for line in head.splitlines():
            if line.startswith("==="):
                meta["title"] = line.strip("= ").strip()
            for key, pat in (("publisher", "Fonte:"), ("url", "URL:"), ("license", "Licenca:")):
                if line.startswith(pat):
                    meta[key] = line.split(":", 1)[1].strip()
            if line.startswith("Doenca:"):
                seg = line.split(":", 1)[1]
                meta["disease"] = seg.split("|")[0].strip()
                if "Idioma:" in line:
                    meta["lang"] = line.split("Idioma:")[1].strip()
    meta["doc_id"] = os.path.splitext(os.path.basename(path))[0]
    if not meta["disease"]:
        meta["disease"] = os.path.basename(os.path.dirname(path))
    return meta, body.strip()

def split_paragraphs(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Corta em pedacos respeitando paragrafos; janela com sobreposicao p/ paragrafos longos."""
    chunks, cur = [], ""
    for para in split_paragraphs(text):
        if len(para) > size:
            if cur:
                chunks.append(cur); cur = ""
            step = max(1, size - overlap)   # evita laco infinito se overlap >= size
            i = 0
            while i < len(para):
                chunks.append(para[i:i + size])
                i += step
        elif len(cur) + len(para) + 2 <= size:
            cur = (cur + "\n\n" + para) if cur else para
        else:
            if cur:
                chunks.append(cur)
            cur = para
    if cur:
        chunks.append(cur)
    return [c.strip() for c in chunks if len(c.strip()) > 60]

# ----------------------------- tokenizacao BM25 -------------------------------
# Reutiliza o MESMO tokenizador do rag_core para o indice e a consulta nunca divergirem.
import sys
sys.path.insert(0, ROOT)
from rag_core import _tokenize as tokenize, _strip_accents as strip_accents  # noqa: E402,F401

# ----------------------------- pipeline ---------------------------------------
def build():
    # determinismo do build (o indice gerado e enviado junto = versao canonica)
    import random
    random.seed(0); np.random.seed(0)
    try:
        import torch; torch.manual_seed(0)
    except Exception:
        pass
    os.makedirs(IDX, exist_ok=True)
    paths = [p for p in glob.glob(os.path.join(KB, "*", "*.txt"))]
    paths.sort()
    records = []
    for path in paths:
        meta, body = read_doc(path)
        for j, ch in enumerate(chunk_text(body)):
            records.append({
                "id": f"{meta['doc_id']}__{j}",
                "doc_id": meta["doc_id"], "disease": meta["disease"],
                "title": meta["title"], "url": meta["url"],
                "license": meta["license"], "lang": meta["lang"],
                "chunk_idx": j, "text": ch,
            })
    print(f"{len(paths)} documentos -> {len(records)} pedacos (chunks)")

    # embeddings (e5 exige prefixo 'passage:' nos documentos)
    from sentence_transformers import SentenceTransformer
    print(f"Carregando modelo de embeddings: {EMB_MODEL} ...")
    model = SentenceTransformer(EMB_MODEL)
    passages = [f"passage: {r['text']}" for r in records]
    emb = model.encode(passages, batch_size=32, show_progress_bar=True,
                       normalize_embeddings=True, convert_to_numpy=True).astype("float32")

    # FAISS (produto interno em vetores normalizados = cosseno)
    import faiss
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, os.path.join(IDX, "faiss.index"))

    # BM25
    from rank_bm25 import BM25Okapi
    tokenized = [tokenize(r["text"]) for r in records]
    bm25 = BM25Okapi(tokenized)
    with open(os.path.join(IDX, "bm25.pkl"), "wb") as f:
        pickle.dump({"bm25": bm25, "tokenized_len": len(tokenized)}, f)

    # chunks + metadados
    with open(os.path.join(IDX, "chunks.jsonl"), "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    np.save(os.path.join(IDX, "embeddings.npy"), emb)
    meta = {"emb_model": EMB_MODEL, "n_docs": len(paths), "n_chunks": len(records),
            "dim": int(emb.shape[1]), "chunk_size": CHUNK_SIZE, "overlap": CHUNK_OVERLAP}
    json.dump(meta, open(os.path.join(IDX, "meta.json"), "w"), indent=2)

    by_disease = {}
    for r in records:
        by_disease[r["disease"]] = by_disease.get(r["disease"], 0) + 1
    print("Chunks por doenca:", by_disease)
    print(f"Indices salvos em {IDX}")

if __name__ == "__main__":
    build()

## 2. Arquitetura
Núcleo em `rag_core.py`; o índice é pré-construído por `scripts/indexar.py` e reusado aqui. O pipeline é
**determinístico** de ponta a ponta: a recuperação exata e a decodificação *greedy* (`do_sample=False`)
tornam os resultados reprodutíveis.
- **Indexação semântica:** os trechos (~1100 caracteres, chunking por parágrafo com sobreposição) são
  codificados por `intfloat/multilingual-e5-base` (prefixos `passage:` e `query:`), normalizados e indexados
  em FAISS **`IndexFlatIP`** (produto interno em vetores normalizados ≡ similaridade de cosseno).
- **Recuperação (as 3 configurações):** compartilham o índice e diferem só na seleção dos trechos:
  (1) **baseline**, sem recuperação; (2) **RAG**, top-k denso; (3) **proposta**, top-k denso ampliado e
  reordenado por um **reranker cross-encoder** (`BAAI/bge-reranker-v2-m3`), que reescora cada par
  (consulta, trecho) e promove o mais pertinente ao topo do contexto.
- **Decisão de projeto:** uma fusão híbrida densa+BM25 via *Reciprocal Rank Fusion* (`hybrid_search`) foi
  implementada e descartada, pois introduzia trechos fora de tópico que degradavam a geração no modelo de 3B;
  densa→reranker preservou a precisão sem o ruído.
- **Geração:** `Qwen2.5-3B-Instruct` (Apache-2.0) servido localmente em fp16 na GPU; a mesma instância atua
  como **LLM-as-judge** na avaliação.
- **O que cada modelo aprendeu (*transfer learning*):** o pipeline compõe modelos **pré-treinados**, sem
  treino próprio. O *embedder* (`e5`) é um **bi-encoder** que aprendeu, por **aprendizado contrastivo**, a
  aproximar textos de sentido próximo; o reranker é um **cross-encoder**, um **classificador de relevância**
  que aprendeu, de pares (consulta, trecho) rotulados, a pontuar quão pertinente é um trecho; o LLM aprendeu
  **modelagem de linguagem**. Bi-encoder é rápido (busca); cross-encoder é preciso (reordenação).

In [ ]:
# Figura metodologica: as 3 configuracoes lado a lado (diferem so na SELECAO dos trechos que vao ao LLM).
from IPython.display import Image
Image(filename="figuras/pipeline_rag.png")

### O núcleo (`rag_core.py`): implementação completa
A célula abaixo **grava** o núcleo do sistema como o módulo `rag_core.py`; as células seguintes o **importam**.
Assim o **código-fonte (comentado) fica visível e autocontido no notebook**, e o mesmo módulo é reusado pelos
scripts. *(A célula apenas escreve o arquivo, não treina nada.)*

In [ ]:
%%writefile rag_core.py
"""
rag_core.py - Nucleo do assistente de pneumologia (RAG).
Implementacao individual (Trabalho Final AM1PIDPG-93, UNESP) - autor: Nycolas Mariotto.

Reune a logica das 3 configuracoes comparadas no trabalho:
  - answer_baseline(q)  : LLM SEM consulta (so de cabeca)
  - answer_rag(q)       : busca densa (FAISS) top-k -> LLM                  [RAG convencional]
  - answer_proposed(q)  : busca densa amplia o pool -> reranker reordena -> LLM [proposta]

A GERACAO usa um modelo ABERTO rodando LOCALMENTE (HuggingFace transformers):
Qwen2.5-3B-Instruct (Apache-2.0, forte em portugues, cabe em ~6 GB fp16 e na GPU T4 do Colab).
Assim o notebook roda 100% no Colab com "Run all" e SEM chave de API. O modelo e configuravel
por ambiente (LLM_MODEL) e, em GPU pequena, LLM_4BIT=1 carrega um modelo MAIOR em 4-bit (usado
na ablacao de tamanho de modelo: 3B vs 7B).

Modelos carregados de forma "preguicosa" (lazy): so baixam/inicializam no 1o uso.
O notebook do Colab importa este mesmo modulo, garantindo que codigo testado == codigo entregue.
Geracao DETERMINISTICA (greedy, do_sample=False) -> resultados reprodutiveis.
"""
import os, re, json, pickle, unicodedata
import numpy as np

# Saida limpa no notebook: silencia avisos/barras de download do HuggingFace (HF_TOKEN opcional,
# barras de progresso, advisories) para nao poluir as respostas da demonstracao. (Definir ANTES
# de importar transformers/huggingface_hub, que sao carregados de forma preguicosa abaixo.)
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings, logging
warnings.filterwarnings("ignore")
for _n in ("huggingface_hub", "huggingface_hub.utils._http", "transformers"):
    logging.getLogger(_n).setLevel(logging.ERROR)

ROOT = os.path.dirname(os.path.abspath(__file__))
IDX = os.path.join(ROOT, "indices")
LLM_MODEL = os.environ.get("LLM_MODEL", "Qwen/Qwen2.5-3B-Instruct")
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "512"))
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
# Embedder/reranker na CPU por padrao: liberam VRAM para o prefill do LLM. Em placa de 8 GB, reranker
# na GPU rouba esse headroom -> geracao de contexto longo ~3x mais lenta (fp32 na CPU tambem da scores
# deterministicos). RERANK_DEVICE=cuda em GPU folgada.
EMB_DEVICE = os.environ.get("EMB_DEVICE", "cpu")
RERANK_DEVICE = os.environ.get("RERANK_DEVICE", "cpu")
# Modelo maior numa GPU pequena: LLM_4BIT=1 carrega em 4-bit (NF4) via bitsandbytes. O padrao (3B)
# usa fp16 e NAO depende de bitsandbytes. Ex.: LLM_MODEL=Qwen/Qwen2.5-7B-Instruct LLM_4BIT=1 (ablacao).
LOAD_4BIT = os.environ.get("LLM_4BIT", "").lower() in ("1", "true", "yes")

# --------------------------------------------------------------------------- #
# Carregamento preguicoso (lazy) de indices e modelos
# --------------------------------------------------------------------------- #
_state = {"emb": None, "rer": None, "faiss": None, "bm25": None, "chunks": None,
          "llm": None, "meta": None, "tok": 0}

def _meta():
    if _state["meta"] is None:
        _state["meta"] = json.load(open(os.path.join(IDX, "meta.json"), encoding="utf-8"))
    return _state["meta"]

def chunks():
    if _state["chunks"] is None:
        _state["chunks"] = [json.loads(l) for l in open(os.path.join(IDX, "chunks.jsonl"), encoding="utf-8")]
    return _state["chunks"]

def _faiss():
    if _state["faiss"] is None:
        import faiss
        _state["faiss"] = faiss.read_index(os.path.join(IDX, "faiss.index"))
    return _state["faiss"]

def _bm25():
    if _state["bm25"] is None:
        _state["bm25"] = pickle.load(open(os.path.join(IDX, "bm25.pkl"), "rb"))["bm25"]
    return _state["bm25"]

def _emb():
    if _state["emb"] is None:
        from sentence_transformers import SentenceTransformer
        _state["emb"] = SentenceTransformer(_meta()["emb_model"], device=EMB_DEVICE)
    return _state["emb"]

def _reranker():
    if _state["rer"] is None:
        import torch
        from sentence_transformers import CrossEncoder
        dev = RERANK_DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
        if dev == "cuda":
            _llm()  # carrega o LLM PRIMEIRO: em placa de 8 GB ele precisa ocupar a GPU inteira antes
                    # do reranker, senao 'device_map=auto' descarrega camadas p/ a CPU (geracao ~3x lenta)
        mk = {"torch_dtype": torch.float16} if dev == "cuda" else {}
        _state["rer"] = CrossEncoder(RERANKER_MODEL, max_length=512, device=dev, model_kwargs=mk)
    return _state["rer"]

def _llm():
    """Carrega o modelo aberto 1x (lazy). Padrao: GPU em fp16 (device_map='auto') ou CPU em fp32
    sem CUDA. Com LLM_4BIT=1, carrega em 4-bit (NF4) - permite rodar um modelo MAIOR (ex.: 7B) numa
    GPU pequena, ao custo de exigir 'bitsandbytes'. Precisa de 'accelerate' instalado."""
    if _state["llm"] is None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        use_cuda = torch.cuda.is_available()
        tok = AutoTokenizer.from_pretrained(LLM_MODEL)
        kw = {"device_map": "auto"}
        if use_cuda and LOAD_4BIT:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        else:
            kw["dtype"] = torch.float16 if use_cuda else torch.float32  # 'dtype' (torch_dtype foi deprecado)
        model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, **kw)
        model.eval()
        _state["llm"] = (tok, model)
    return _state["llm"]

# --------------------------------------------------------------------------- #
# Busca (retrieval) - tudo local e deterministico, sem chave
# --------------------------------------------------------------------------- #
def _strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def _tokenize(text):
    return re.findall(r"[a-z0-9]+", _strip_accents(text.lower()))

def dense_search(query, k=10):
    """Busca por significado: embedding da pergunta (prefixo 'query:' do e5) vs FAISS."""
    q = _emb().encode([f"query: {query}"], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    D, I = _faiss().search(q, k)
    return [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i >= 0]

def bm25_search(query, k=10):
    """Busca por palavra-chave (BM25)."""
    scores = _bm25().get_scores(_tokenize(query))
    order = np.argsort(-scores, kind="stable")[:k]  # estavel: empates por indice (reproduzivel)
    return [(int(i), float(scores[i])) for i in order]

def _rrf(rankings, k=60):
    """Reciprocal Rank Fusion: combina varias listas ordenadas em um ranking unico."""
    acc = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            acc[idx] = acc.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(acc, key=acc.get, reverse=True)

def hybrid_search(query, k=10, pool=20):
    """Combina busca densa + BM25 via RRF."""
    d = [i for i, _ in dense_search(query, pool)]
    b = [i for i, _ in bm25_search(query, pool)]
    return _rrf([d, b])[:k]

def rerank(query, cand_idx, top_n=5):
    """Reordena candidatos com cross-encoder (leitor criterioso)."""
    if not cand_idx:
        return []
    ch = chunks()
    pairs = [[query, ch[i]["text"]] for i in cand_idx]
    scores = np.asarray(_reranker().predict(pairs, batch_size=16))  # batch limitado: pico de VRAM seguro
    order = np.argsort(-scores, kind="stable")[:top_n]  # estavel: empates por indice
    return [(cand_idx[o], float(scores[o])) for o in order]

# --------------------------------------------------------------------------- #
# PROMPTS DE SISTEMA (a persona/instrucoes que cada configuracao passa ao modelo)
# --------------------------------------------------------------------------- #
# Persona forte e IGUAL no baseline e no RAG (justo: entre eles so a RECUPERACAO muda).
SYS_BASE = ("Você é um médico pneumologista experiente, com sólido domínio de medicina baseada "
            "em evidências. Responda à pergunta clínica de forma objetiva, tecnicamente precisa e "
            "concisa, em português do Brasil, priorizando o que é consenso na literatura. Se não "
            "tiver certeza, diga que não sabe em vez de inventar.")
SYS_RAG = ("Você é um médico pneumologista experiente, com sólido domínio de medicina baseada em "
           "evidências. Use PRIORITARIAMENTE os TRECHOS fornecidos e cite a fonte como [Fonte N] "
           "sempre que a informação vier deles. Se algum detalhe não estiver nos trechos, complemente "
           "com conhecimento clínico consolidado, deixando claro o que veio das fontes. Não invente "
           "fontes nem dados. Responda de forma objetiva, tecnicamente precisa e concisa, em "
           "português do Brasil.")
# Sistema do juiz = papel + rigor. A REGUA (notas 0/1/2, JSON) vai na mensagem de USUARIO, montada
# em avaliar.py (judge_one) -> e a ela que "as instrucoes fornecidas a seguir" se referem.
SYS_JUDGE = ("Você é um avaliador médico rigoroso e objetivo. Siga exatamente as instruções de "
             "avaliação fornecidas a seguir.")

def reset_tokens():
    _state["tok"] = 0

def get_tokens():
    return _state["tok"]

def _complete(system, user, max_new_tokens=None):
    """Geracao local DETERMINISTICA (greedy). Aplica o chat template (system+user) do modelo.
    Retorna (texto, n_tokens) onde n_tokens = prompt + resposta (proxy de custo).
    `max_new_tokens` permite respostas curtas (ex.: juiz que so devolve um numero)."""
    import torch
    tok, model = _llm()
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(next(model.parameters()).device)
    n_in = inputs["input_ids"].shape[1]
    mnt = MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=mnt, do_sample=False,
                             pad_token_id=(tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id))
    new = out[0][n_in:]
    text = tok.decode(new, skip_special_tokens=True).strip()
    return text, int(n_in + new.shape[0])

def _gen(system, user):
    text, n = _complete(system, user)
    _state["tok"] += n
    return text

def raw_generate(prompt, max_new_tokens=None):
    """Geracao sem contabilizar tokens (usada pela IA-juiza na avaliacao)."""
    return _complete(SYS_JUDGE, prompt, max_new_tokens=max_new_tokens)[0]

def _context(idxs):
    # [Fonte N] numerado por DOCUMENTO (nao por chunk): chunks do mesmo doc compartilham o numero,
    # entao a citacao [Fonte N] no texto bate 1:1 com a lista de fontes (deduplicada por doc) exibida.
    ch = chunks()
    num, blocks = {}, []
    for i in idxs:
        c = ch[i]
        d = c["doc_id"]
        if d not in num:
            num[d] = len(num) + 1
        blocks.append(f"[Fonte {num[d]}: {c['title']} — {c['disease']}]\n{c['text']}")
    return "\n\n".join(blocks)

def _sources(idxs):
    ch = chunks()
    seen, out = set(), []
    for i in idxs:
        key = ch[i]["doc_id"]
        if key not in seen:
            seen.add(key)
            out.append({"title": ch[i]["title"], "disease": ch[i]["disease"], "url": ch[i]["url"]})
    return out

# =========================================================================== #
# AS 3 CONFIGURACOES COMPARADAS NO TRABALHO
# O que muda entre elas e SO a selecao dos trechos que chegam ao MESMO LLM:
#   CONFIG 1 (answer_baseline) : LLM sozinho, SEM recuperacao (so conhecimento parametrico)
#   CONFIG 2 (answer_rag)      : busca densa (FAISS) top-k -> LLM                [RAG convencional]
#   CONFIG 3 (answer_proposed) : busca densa (pool) -> RERANKER reordena -> LLM  [proposta]
# =========================================================================== #

# ---- CONFIG 1: BASELINE (sem recuperacao) ----
def answer_baseline(question):
    """Config 1: sem consulta - o LLM responde so de cabeca (conhecimento parametrico)."""
    out = _gen(SYS_BASE, f"Pergunta: {question}\n\nResposta:")
    return {"answer": out, "sources": [], "context_idx": []}

# ---- CONFIG 2: RAG CONVENCIONAL (busca densa -> LLM) ----
def answer_rag(question, k=5, system=SYS_RAG):
    """Config 2 (RAG convencional): busca densa top-k -> LLM. Sem reranker.
    `system` default = SYS_RAG (config oficial); aceita um prompt alternativo (usado em experimentos)."""
    idxs = [i for i, _ in dense_search(question, k)]
    user = f"TRECHOS:\n{_context(idxs)}\n\nPergunta: {question}\n\nResposta:"
    return {"answer": _gen(system, user), "sources": _sources(idxs), "context_idx": idxs}

# ---- CONFIG 3: PROPOSTA (busca densa -> reranker -> LLM) ----
def answer_proposed(question, k=5, pool=40, system=SYS_RAG):
    """Config 3 (proposta): a busca densa amplia o pool de candidatos (top-`pool`) e um RERANKER
    cross-encoder (leitor criterioso) os reordena por RELEVANCIA real, ficando com os `k` melhores
    -> LLM. O reranker neural e a tecnica avancada alem do RAG simples: recuperacao mais PRECISA,
    com o trecho-chave no topo. (BM25/hibrido tambem foram implementados - ver hybrid_search -, mas
    injetavam trechos fora do tema que confundiam o modelo pequeno; a densa+reranker ficou mais limpa.)"""
    cand = [i for i, _ in dense_search(question, pool)]
    idxs = [i for i, _ in rerank(question, cand, top_n=k)]
    user = f"TRECHOS:\n{_context(idxs)}\n\nPergunta: {question}\n\nResposta:"
    return {"answer": _gen(system, user), "sources": _sources(idxs), "context_idx": idxs}

# --------------------------------------------------------------------------- #
# Experimento H1 (opcional) - HyDE: Hypothetical Document Embeddings (Gao et al., 2022)
# Isolado das 3 configuracoes oficiais; nao altera baseline/rag/proposta.
# --------------------------------------------------------------------------- #
SYS_HYDE = ("Voce e um pneumologista. Escreva UM paragrafo curto (2 a 4 frases), tecnico e direto, "
            "como se fosse um trecho de diretriz que responde a pergunta. Nao cite fontes nem use rodeios.")

def hyde_doc(question):
    """Gera o 'documento hipotetico': uma resposta plausivel em linguagem de diretriz. Esse texto fica
    mais proximo dos TRECHOS reais do que a pergunta curta, melhorando o casamento na busca densa."""
    return _gen(SYS_HYDE, f"Pergunta: {question}\n\nTrecho:")

def dense_search_hyde(query, k=10):
    """Busca densa usando o documento hipotetico como consulta. Ele e embedado no MESMO espaco dos
    documentos (prefixo 'passage:' do e5), pois faz papel de documento, nao de pergunta."""
    hd = hyde_doc(query)
    v = _emb().encode([f"passage: {hd}"], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    D, I = _faiss().search(v, k)
    return [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i >= 0]

def answer_hyde(question, k=5, system=SYS_RAG):
    """Experimento H1 (HyDE): o modelo redige um documento hipotetico, ele vira a consulta densa, e os
    trechos recuperados vao ao LLM. Mesmo gerador/leitor das outras configuracoes, so muda a consulta."""
    idxs = [i for i, _ in dense_search_hyde(question, k)]
    user = f"TRECHOS:\n{_context(idxs)}\n\nPergunta: {question}\n\nResposta:"
    return {"answer": _gen(system, user), "sources": _sources(idxs), "context_idx": idxs}

In [ ]:
import rag_core as rc
print("Modelo de geração (local):", rc.LLM_MODEL)
print("Funções: rc.answer_baseline(q) | rc.answer_rag(q) | rc.answer_proposed(q)")

## 3. Demonstração ao vivo
As 3 configurações sobre a mesma pergunta, um caso de **domínio regional** (paracoccidioidomicose),
onde o conhecimento paramétrico do modelo falha e a recuperação é decisiva. (A 1ª geração baixa e
carrega o modelo, ~6 GB; as seguintes são rápidas.)

In [ ]:
pergunta = "Qual o tratamento de primeira escolha das formas leves a moderadas de PCM?"
for nome, fn in [("1) BASELINE: sem recuperação", rc.answer_baseline),
                 ("2) RAG: recuperação densa", rc.answer_rag),
                 ("3) PROPOSTA: densa + reranking", rc.answer_proposed)]:
    r = fn(pergunta)
    print("=" * 90)
    print(nome)
    print(r["answer"])
    if r["sources"]:
        print("Fontes:")
        for n, s in enumerate(r["sources"], 1):
            print(f"   [Fonte {n}] {s['title']}  ·  {s['disease']}")

### O que o modelo realmente recebe
Na configuração RAG, o modelo não vê só a pergunta: recebe o **prompt de sistema** (persona) + os
**trechos recuperados** + a pergunta. A célula abaixo imprime esse prompt montado para a pergunta da demo,
tornando o mecanismo do RAG explícito.

In [ ]:
# Mostra o prompt REAL da config RAG: sistema (SYS_RAG) + TRECHOS recuperados + pergunta.
q_demo = "Qual o tratamento de primeira escolha das formas leves a moderadas de PCM?"
idxs = [i for i, _ in rc.dense_search(q_demo, 5)]
print("=== PROMPT DE SISTEMA (SYS_RAG) ===")
print(rc.SYS_RAG)
print()
print("=== MENSAGEM DE USUÁRIO (trechos recuperados + pergunta) ===")
user = f"TRECHOS:\n{rc._context(idxs)}\n\nPergunta: {q_demo}\n\nResposta:"
print(user[:1500] + ("\n[...]" if len(user) > 1500 else ""))

## 4. Avaliação comparativa (54 perguntas)
O benchmark são **54 perguntas PT-BR que eu montei**, de fácil a difícil, cada uma com uma
**resposta-gabarito** e a **fonte-ouro** de onde ela vem. Resultados **pré-computados** sobre esse conjunto
(rodar ao vivo leva ~30 min na GPU; `scripts/avaliar.py` reproduz). Métricas: **qualidade da resposta**
(LLM-as-judge, nota 0–2 vs. gabarito), **fidelidade** (a resposta se ancora nos trechos? padrão RAGAS),
**taxa de citação** de fontes, **recuperação** (hit-rate e MRR contra a fonte-ouro) e **custo** (latência,
tokens). *Hit-rate* = a fonte-ouro apareceu entre os trechos recuperados; *MRR* = quão perto do topo ela veio.

In [ ]:
# As 54 perguntas do benchmark: cada uma com pergunta, gabarito e a fonte-ouro de onde vem.
import json
try:
    import pandas as pd
    from IPython.display import display
    _PD = True
except Exception:
    _PD = False
bench = json.load(open("benchmark/perguntas.json", encoding="utf-8"))["perguntas"]
print(f"Total: {len(bench)} perguntas | por doenca:",
      {d: sum(q["disease"] == d for q in bench) for d in ("dpoc", "fibrose", "pcm")},
      "| por dificuldade:",
      {x: sum(q["dificuldade"] == x for q in bench) for x in ("facil", "media", "dificil")})
def _corta(s, n=90):
    s = s or ""
    return (s[:n] + "...") if len(s) > n else s
linhas = [{"id": q["id"], "doenca": q["disease"], "dif.": q["dificuldade"],
           "pergunta": _corta(q["pergunta"]), "gabarito": _corta(q["resposta_referencia"]),
           "fonte-ouro": q["fonte_id"]} for q in bench]
if _PD:
    display(pd.DataFrame(linhas).set_index("id"))
else:
    for r in linhas:
        print(r["id"], "|", r["doenca"], "|", r["dif."], "|", r["pergunta"])

### O harness de avaliação (`scripts/avaliar.py`)
O código abaixo roda as 3 configurações em cada pergunta e mede tudo, usando o **mesmo modelo local** como
**juiz** (LLM-as-judge). **Já foi rodado** — os resultados prontos vêm no repositório (rodar de novo leva
~30 min na GPU). A célula grava o arquivo para deixar a lógica visível, mas **não o executa aqui**.

In [ ]:
%%writefile scripts/avaliar.py
"""
Avaliacao comparativa das 3 configuracoes (baseline / RAG / proposta).
Para cada pergunta: roda as 3 versoes e mede
  - qualidade (IA-juiza compara cada resposta ao gabarito: nota 0/1/2)
  - fidelidade (groundedness, padrao RAGAS: a resposta se ancora nos trechos recuperados?)
  - acerto da busca (a fonte certa entrou no contexto? -> hit-rate e MRR) [rag e proposta]
  - custo (latencia REAL em s, tokens prompt+resposta do modelo local)
GERACAO E JUIZA usam o MESMO modelo aberto LOCAL (Qwen2.5 via transformers) - sem chave, sem cota,
deterministico (greedy). RESUMIVEL: salva o progresso a cada pergunta em _resultados/aval_<modo>.json;
um crash no meio continua de onde parou. Gera grafico no final. O modelo e configuravel por ambiente
(LLM_MODEL / LLM_4BIT); MODEL_TAG separa os arquivos de saida (ex.: 3b vs 7b na ablacao).

Uso:  python avaliar.py all   (54 perguntas)  |  python avaliar.py preview  (12)
"""
import os, sys, json, time, re
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import rag_core as rc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
OUT = os.path.join(ROOT, "_resultados")
os.makedirs(OUT, exist_ok=True)

PREVIEW = ["dpoc_02", "dpoc_08", "dpoc_10", "dpoc_13", "fib_06", "fib_07",
           "fib_09", "fib_13", "pcm_04", "pcm_08", "pcm_11", "pcm_14"]
CFGS = [("baseline", rc.answer_baseline), ("rag", rc.answer_rag), ("proposta", rc.answer_proposed)]

def _save(path, obj):
    """Escrita atomica (.tmp -> replace): um crash no meio nao corrompe o checkpoint."""
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def _parse_nota(raw):
    """Extrai a nota 0/1/2 da resposta da juiza. ANCORA em 'nota' (cobre {"nota": 2}, 'Nota 1',
    'nota=0'); so se nao houver, usa o ULTIMO 0/1/2 isolado (o veredito costuma vir por ultimo) -
    evita pegar um digito do raciocinio/conteudo (ex.: '2 criterios', '0,70')."""
    raw = raw or ""
    ms = re.findall(r'nota["\s:=]*([012])', raw, re.IGNORECASE)  # ULTIMO match: o veredito vem por
    if ms:                                                        # ultimo, apos qualquer eco do molde
        return int(ms[-1])
    achados = re.findall(r"\b([012])\b", raw)
    return int(achados[-1]) if achados else None

def judge_one(pergunta, gabarito, answer):
    """Avalia UMA resposta vs gabarito (pontual, CEGA a qual configuracao gerou -> sem vies de
    posicao). Cada resposta e julgada pelo seu proprio merito contra a referencia."""
    p = ("Você é um avaliador médico rigoroso. Compare a RESPOSTA ao GABARITO e atribua UMA nota: "
         "2 = correta e suficiente (contém o ponto principal do gabarito, mesmo que com texto extra); "
         "1 = parcialmente correta (cita o ponto certo mas incompleta, ou com erro secundário); "
         "0 = incorreta, vazia ou recusa a responder.\n\n"
         f"PERGUNTA: {pergunta}\nGABARITO: {gabarito}\n\nRESPOSTA:\n{answer}\n\n"
         'Responda APENAS em JSON: {"nota": 0, 1 ou 2}')
    nota = _parse_nota(rc.raw_generate(p))
    if nota is None:
        print("  [aviso] juiza nao retornou nota valida; assumindo 0", flush=True)
        return 0
    return nota

def judge_faithfulness(answer, context):
    """FIDELIDADE (groundedness, padrao RAGAS): a RESPOSTA usa a informacao dos TRECHOS recuperados?
    Formulacao simples e direta -- a versao rigida ('APENAS em JSON' + regua dura) colapsava o juiz
    de 3B para 0 ate em respostas ancoradas. Nota 0/1/2. Sem contexto (baseline) -> 0."""
    if not (context or "").strip():
        return 0
    p = ("Os TRECHOS abaixo foram fornecidos a um assistente médico. A RESPOSTA usa a informação "
         "desses trechos? 2 = a maior parte da resposta se apoia nos trechos; 1 = em parte; "
         "0 = a resposta ignora os trechos ou os contradiz.\n\n"
         f"TRECHOS:\n{context}\n\nRESPOSTA:\n{answer}\n\n"
         "Responda só com o número (0, 1 ou 2). Nota:")
    n = _parse_nota(rc.raw_generate(p))
    return 0 if n is None else n

def load_done(path):
    if os.path.exists(path):
        try:
            return {r["id"]: r for r in json.load(open(path, encoding="utf-8")).get("rows", [])}
        except Exception:
            return {}
    return {}

def run(qs, path):
    ch = rc.chunks()
    done = load_done(path)
    rows = [done[i] for i in done]
    if len(done) < len(qs):  # aquece modelos (LLM+embedder+reranker) p/ latencias limpas (sem o load 1x)
        print("  aquecendo modelos (1a carga)...", flush=True)
        rc.answer_proposed("aquecimento do sistema")
    for q in qs:
        if q["id"] in done:
            print(f"  (ja feito) {q['id']}")
            continue
        item = {"id": q["id"], "disease": q["disease"], "dificuldade": q["dificuldade"], "fonte_id": q["fonte_id"]}
        ans = {}
        for cfg, fn in CFGS:
            rc.reset_tokens()
            t0 = time.time()
            r = fn(q["pergunta"])
            lat = time.time() - t0
            tok = rc.get_tokens()
            docs = [ch[i]["doc_id"] for i in r["context_idx"]]
            uniq = list(dict.fromkeys(docs))            # docs unicos na ordem -> MRR por doc, nao por chunk
            hit = q["fonte_id"] in uniq
            rank = uniq.index(q["fonte_id"]) + 1 if hit else 0
            ctx = rc._context(r["context_idx"]) if r["context_idx"] else ""   # texto dos trechos recuperados
            fid = judge_faithfulness(r["answer"], ctx)   # fidelidade: a resposta se ancora nos trechos?
            item[cfg] = {"lat": round(lat, 2), "tok": tok, "hit": hit, "rank": rank, "fidelidade": fid,
                         "answer": r["answer"], "docs": docs}
            ans[cfg] = r["answer"]
            print(f"  {q['id']:<9} {cfg:<9} {lat:5.1f}s {tok:>5}tok hit={hit} fid={fid}", flush=True)
        notas = {cfg: judge_one(q["pergunta"], q["resposta_referencia"], ans[cfg])
                 for cfg in ("baseline", "rag", "proposta")}
        for cfg in ("baseline", "rag", "proposta"):
            item[cfg]["nota"] = notas[cfg]
        print(f"    -> notas {notas}", flush=True)
        rows.append(item)
        _save(path, {"rows": rows})
    return rows

def aggregate(rows):
    if not rows:
        return {}
    out = {}
    for c, _ in CFGS:
        notas = [r[c]["nota"] for r in rows]
        d = {"qualidade": sum(notas) / (2 * len(notas)),
             "pct_correto": sum(n == 2 for n in notas) / len(notas),
             "fidelidade": sum(r[c].get("fidelidade", 0) for r in rows) / (2 * len(rows)),  # 0/1/2 -> 0-1
             "lat_med": sum(r[c]["lat"] for r in rows) / len(rows),
             "tok_med": sum(r[c]["tok"] for r in rows) / len(rows)}
        if c in ("rag", "proposta"):
            d["hit_rate"] = sum(r[c]["hit"] for r in rows) / len(rows)
            d["mrr"] = sum((1.0 / r[c]["rank"]) for r in rows if r[c]["rank"] > 0) / len(rows)
        out[c] = d
    return out

def graficos(agg, n, tag):
    labels = ["Baseline", "RAG", "Proposta"]
    cols = ["#9aa0a6", "#4285F4", "#34A853"]
    order = ("baseline", "rag", "proposta")
    fig, ax = plt.subplots(1, 4, figsize=(17, 4))
    ax[0].bar(labels, [agg[c]["qualidade"] for c in order], color=cols)
    ax[0].set_title("Qualidade da resposta (0-1)\nIA-juiza vs gabarito"); ax[0].set_ylim(0, 1)
    ax[1].bar(labels, [agg[c]["fidelidade"] for c in order], color=cols)
    ax[1].set_title("Fidelidade (0-1)\nresposta ancorada nos trechos"); ax[1].set_ylim(0, 1)
    ax[2].bar(["RAG", "Proposta"], [agg["rag"]["hit_rate"], agg["proposta"]["hit_rate"]], color=cols[1:])
    ax[2].set_title("Acerto da busca (hit-rate)\nfonte certa no contexto"); ax[2].set_ylim(0, 1)
    ax[3].bar(labels, [agg[c]["lat_med"] for c in order], color=cols)
    ax[3].set_title("Latencia media (s)")
    for a in ax:
        for p in a.patches:
            a.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                       ha="center", va="bottom", fontsize=9)
    fig.suptitle(f"Comparacao Baseline x RAG x Proposta  ({tag}, n={n})", fontweight="bold")
    fig.tight_layout()
    path = os.path.join(OUT, f"comparacao_{tag}.png")
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print("Grafico salvo:", path)

if __name__ == "__main__":
    modo = sys.argv[1] if len(sys.argv) > 1 else "all"
    bm = json.load(open(os.path.join(ROOT, "benchmark", "perguntas.json"), encoding="utf-8"))["perguntas"]
    qs = bm if modo == "all" else [q for q in bm if q["id"] in PREVIEW]
    tag = os.environ.get("MODEL_TAG", "")   # ex.: MODEL_TAG=3b / 7b p/ separar os arquivos da ablacao
    path = os.path.join(OUT, f"aval_{modo}{('_' + tag) if tag else ''}.json")
    print(f"Avaliando {len(qs)} perguntas (modo={modo}, modelo={rc.LLM_MODEL})...\n")
    rows = run(qs, path)
    if not rows:
        print("Nenhuma pergunta avaliada."); sys.exit(0)
    agg = aggregate(rows)
    _save(path, {"modo": modo, "n": len(rows), "agg": agg, "rows": rows})
    graficos(agg, len(rows), f"{modo}{('_' + tag) if tag else ''}")
    print("\n=== RESUMO ===")
    for c, _ in CFGS:
        a = agg[c]
        extra = f" | busca: hit={a.get('hit_rate', 0):.0%} MRR={a.get('mrr', 0):.2f}" if "hit_rate" in a else ""
        print(f"{c:<9} qualidade={a['qualidade']:.2f} fidelidade={a['fidelidade']:.2f} "
              f"correto={a['pct_correto']:.0%} lat={a['lat_med']:.1f}s tok={a['tok_med']:.0f}{extra}")

### O juiz (LLM-as-judge): como a qualidade é medida
A **qualidade** é dada pelo **mesmo modelo** atuando como juiz, de forma **pontual e cega**: cada resposta é
comparada ao gabarito **sem saber qual configuração a gerou** (evita viés de posição). O juiz recebe **duas
mensagens** (system + user):

**Sistema** (`SYS_JUDGE`, em `rag_core.py`):
> *"Você é um avaliador médico rigoroso e objetivo. Siga exatamente as instruções de avaliação fornecidas a seguir."*

**Usuário** (a régua propriamente dita, montada em `judge_one` no `avaliar.py`):
> *"Compare a RESPOSTA ao GABARITO e atribua UMA nota: **2** = correta e suficiente (contém o ponto principal
> do gabarito, mesmo com texto extra); **1** = parcialmente correta (cita o ponto certo mas incompleta, ou
> com erro secundário); **0** = incorreta, vazia ou recusa. Responda APENAS em JSON: {"nota": 0, 1 ou 2}"*

A nota reportada é a **média**, reescalada para **0–1**. A **fidelidade** usa a mesma mecânica (o juiz checa
se a resposta se apoia nos trechos recuperados). ⚠️ Como o juiz é o mesmo modelo que gera, pode haver **viés
de auto-preferência**, por isso a escala é grosseira (0/1/2, robusta a ruído) e as conclusões recomendam
**conferência humana** de um subconjunto.

In [ ]:
%%writefile scripts/resumo_resultados.py
"""
Resumo dos resultados da avaliacao: legenda + geral + por doenca + por dificuldade.

COMO A NOTA E CALCULADA:
O LLM-as-judge compara CADA resposta com o gabarito e da uma nota bruta:
  0 = incorreta, vazia ou recusa  |  1 = parcialmente correta  |  2 = correta e suficiente.
A coluna 'qualidade (0-1)' e a MEDIA dessas notas, normalizada (dividida por 2) para a escala 0 a 1.
  'correto (%)'    = fracao de respostas que receberam a nota maxima (2).
  'cita fonte (%)' = fracao de respostas que citam a fonte no formato [Fonte N] (so as configs com recuperacao).
  'hit-rate (%)' / 'MRR' = a fonte-ouro entrou nos trechos recuperados, e quao perto do topo.
  'tempo (s)' / 'tokens' = custo medio por pergunta.
"""
import os, json, re
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# Arquivo de resultados (default = 3B); AVAL_FILE=aval_all_7b.json reusa esta mesma tabela na ablacao.
AVAL_FILE = os.environ.get("AVAL_FILE", "aval_all.json")
rows = json.load(open(os.path.join(ROOT, "_resultados", AVAL_FILE), encoding="utf-8"))["rows"]

CFGS = ["baseline", "rag", "proposta"]
NAMES = {"baseline": "Baseline", "rag": "RAG convencional", "proposta": "Proposta"}
SHORT = {"baseline": "Baseline", "rag": "RAG", "proposta": "Proposta"}
DIS = {"dpoc": "DPOC", "fibrose": "Fibrose", "pcm": "PCM"}
DIF = {"facil": "Facil", "media": "Media", "dificil": "Dificil"}

def cita(ans): return bool(re.search(r"\[Fonte", ans or ""))
def qual(sub, c): return sum(r[c]["nota"] for r in sub) / (2 * len(sub)) if sub else 0.0

# ---- monta as tabelas (lista de dicts) ----
geral = []
for c in CFGS:
    notas = [r[c]["nota"] for r in rows]
    d = {"Versao": NAMES[c],
         "qualidade (0-1)": round(sum(notas) / (2 * len(notas)), 2),
         "fidelidade (0-1)": round(sum(r[c].get("fidelidade", 0) for r in rows) / (2 * len(rows)), 2),
         "correto (%)": round(100 * sum(n == 2 for n in notas) / len(notas)),
         "cita fonte (%)": round(100 * sum(cita(r[c]["answer"]) for r in rows) / len(rows)),
         "hit-rate (%)": "-", "MRR": "-",
         "tempo (s)": round(sum(r[c]["lat"] for r in rows) / len(rows)),
         "tokens": round(sum(r[c]["tok"] for r in rows) / len(rows))}
    if c in ("rag", "proposta"):
        d["hit-rate (%)"] = round(100 * sum(r[c]["hit"] for r in rows) / len(rows))
        d["MRR"] = round(sum((1 / r[c]["rank"]) for r in rows if r[c]["rank"] > 0) / len(rows), 2)
    geral.append(d)

def tabela_por(campo, mapa, collabel):
    out = []
    for k, nome in mapa.items():
        sub = [r for r in rows if r[campo] == k]
        if not sub:
            continue
        row = {collabel: nome, "n": len(sub)}
        for c in CFGS:
            row[SHORT[c]] = round(qual(sub, c), 2)
        out.append(row)
    return out

por_doenca = tabela_por("disease", DIS, "Doenca")
por_dif = tabela_por("dificuldade", DIF, "Dificuldade")

# ---- exibe: pandas (tabela bonita no Colab) ou texto ----
try:
    import pandas as pd
    from IPython.display import display
    USE_PD = True
except Exception:
    USE_PD = False

print(f"Metricas (calculadas sobre as {len(rows)} perguntas):")
print("  qualidade (0-1)  : media das notas do LLM-as-judge, reescalada para 0-1 (notas brutas: 0 = incorreta, 1 = parcial, 2 = correta)")
print("  fidelidade (0-1) : a resposta se ancora nos trechos recuperados? (groundedness, padrao RAGAS; baseline sem trechos = 0)")
print("  correto (%)      : fracao de respostas com a nota maxima (2)")
print("  cita fonte (%)   : fracao de respostas que citam a fonte como [Fonte N]")
print("  hit-rate / MRR   : com que frequencia (e quao perto do topo) a fonte-ouro entra nos trechos recuperados")
print()

def show(titulo, tabela, indice):
    print(f"=== {titulo} ===")
    if USE_PD:
        display(pd.DataFrame(tabela).set_index(indice))
    else:
        cols = list(tabela[0].keys())
        w = {c: max(len(str(c)), max(len(str(r[c])) for r in tabela)) for c in cols}
        print("  " + "  ".join(str(c).ljust(w[c]) for c in cols))
        for r in tabela:
            print("  " + "  ".join(str(r[c]).ljust(w[c]) for c in cols))
    print()

show(f"GERAL ({len(rows)} perguntas)", geral, "Versao")
show("QUALIDADE (0-1) POR DOENCA", por_doenca, "Doenca")
show("QUALIDADE (0-1) POR DIFICULDADE", por_dif, "Dificuldade")

In [ ]:
# tabela-resumo (geral + por doença + por dificuldade) a partir dos resultados pré-computados
%run scripts/resumo_resultados.py

In [ ]:
# figura comparativa (qualidade, fidelidade, hit-rate e latência)
from IPython.display import Image
import os
_fig = "_resultados/comparacao_all.png" if os.path.exists("_resultados/comparacao_all.png") else "_resultados/comparacao_final.png"
Image(filename=_fig)

## 5b. Ablação de tamanho de modelo (3B vs 7B)
O **achado honesto** (o ganho da recuperação não chega à geração) foi observado no **3B**. Para testar se
ele **se propaga com escala**, rodei a mesma avaliação num modelo maior (**Qwen2.5-7B**, em 4-bit). A célula
abaixo compara os dois: se a *Proposta* passar a superar o *RAG* na qualidade com o 7B, o ganho do reranking
se propaga com escala.

In [ ]:
# Ablacao: compara os resultados agregados de 3B (aval_all.json) e 7B (aval_all_7b.json), se existirem.
import json, os
def _agg(fn):
    p = os.path.join("_resultados", fn)
    return json.load(open(p, encoding="utf-8"))["agg"] if os.path.exists(p) else None
def _linha(tag, a):
    if not a:
        return f"{tag:<4}: (nao rodado ainda)"
    return (f"{tag:<4}: qualidade  RAG={a['rag']['qualidade']:.2f}  Proposta={a['proposta']['qualidade']:.2f}"
            f"   |   hit-rate  RAG={a['rag']['hit_rate']:.0%}  Proposta={a['proposta']['hit_rate']:.0%}"
            f"   |   fidelidade  RAG={a['rag'].get('fidelidade', 0):.2f}  Proposta={a['proposta'].get('fidelidade', 0):.2f}")
print(_linha("3B", _agg("aval_all.json")))
print(_linha("7B", _agg("aval_all_7b.json")))

## 5. Conclusões, limitações e trabalhos futuros
> Este foi um estudo **experimental** de recuperação + geração sobre modelos **pré-treinados**
> (*transfer learning*): a contribuição não é treinar um modelo novo, mas a **comparação controlada** das
> 3 configurações e a **técnica avançada** (reranking), avaliadas com baselines e métricas casadas ao
> problema (recuperação, qualidade da resposta, fidelidade, citação de fontes, custo).

*(Os números abaixo são lidos das tabelas/figuras acima; conferir após cada re-execução.)*

- **A recuperação supera o baseline** na qualidade, com ganho concentrado nas doenças de **menor cobertura
  paramétrica** (fibrose, PCM); além disso, só as configurações com RAG **citam a fonte** e têm **fidelidade
  alta** — o baseline responde "de cabeça" e chega a **alucinar** (confundiu a sigla PCM), com fidelidade baixa.
  Rastreabilidade + fidelidade são essenciais em medicina.
- A **proposta** (densa + **reranking**) **vence na recuperação** (hit-rate/MRR): o cross-encoder cumpre o
  que promete, recuperando com mais precisão.
- **Achado principal (ablação de escala):** no **3B**, o ganho de recuperação da proposta **não se propaga**
  para a geração (proposta ≈ RAG na qualidade) -- um LLM pequeno dilui o contexto mais rico. No **7B**, com a
  **mesma recuperação**, o ganho **aparece**: a proposta passa a superar o RAG em qualidade e em respostas
  corretas (ver §5b). O reranking se traduz em respostas melhores quando o modelo é grande o suficiente.
- **Limitações**: o **LLM-as-judge é o mesmo modelo** (possível **viés de auto-preferência**) → conferência
  humana de um subconjunto; o hit-rate compara com **uma** fonte-ouro por pergunta (a KB tem documentos
  sobrepostos, então pode subestimar); reprodutível na mesma GPU (greedy/determinístico; entre hardwares, fp16
  pode variar pouco).
- **Futuro**: recuperação agêntica (multi-passo), métricas de fidelidade mais ricas e validação clínica com
  especialistas.

**Fontes e licenças:** `FONTES.md`. **Código:** todo embutido acima (`rag_core`, `indexar`, `avaliar`, `resumo`).

## Referências técnicas
- **RAG:** Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020.
- **Embeddings (e5):** Wang et al., *Text Embeddings by Weakly-Supervised Contrastive Pre-training*, 2022.
- **Reranking (cross-encoder):** Nogueira & Cho, *Passage Re-ranking with BERT*, 2019 · reranker `BAAI/bge-reranker-v2-m3`.
- **HyDE:** Gao et al., *Precise Zero-Shot Dense Retrieval without Relevance Labels*, 2022.
- **Fidelidade (groundedness):** Es et al., *RAGAS: Automated Evaluation of Retrieval Augmented Generation*, 2023.
- **Modelo de geração:** Qwen2.5 (Qwen Team, 2024). **Busca vetorial:** FAISS (Johnson et al., 2019).

## 6. Experimento (extra): HyDE (consulta por documento hipotético)
*Exploração adicional, **isolada das 3 configurações** acima (não altera baseline/RAG/proposta).*

Perguntas clínicas curtas ("O que é PCM?") casam mal com a redação dos trechos. **HyDE**
(*Hypothetical Document Embeddings*, Gao et al., 2022) contorna isso: o próprio LLM redige um
**documento hipotético** (uma resposta plausível, em estilo de diretriz) e é *esse* texto, mais
próximo da linguagem do corpus, que vira a consulta densa. Abaixo medimos só o **acerto da busca**
(a fonte-ouro entra no contexto?) nas 54 perguntas, comparando a recuperação densa simples com a via HyDE.
Isso explora justamente a direção de *"recuperação aprimorada"* apontada nas conclusões.

> **Resultado:** no **benchmark completo (54 perguntas)** o HyDE **empata** com a recuperação densa
> (mesmo hit-rate, 30%); ele recupera trechos diferentes, mas não melhora o acerto global. O documento
> hipotético de um modelo de 3B às vezes **desvia do tema**, o que limita o ganho nesta base. Fica como
> exploração, não como melhoria adotada.

In [ ]:
# Experimento opcional: HyDE vs. recuperacao densa simples no benchmark COMPLETO (acerto da busca).
# Isolado — usa rc.dense_search (config RAG) e rc.dense_search_hyde; nao toca as 3 configs.
# O hit-rate nas 54 e pre-computado (rodar o HyDE nas 54 sao 54 geracoes de doc hipotetico, ~10 min);
# aqui exibimos esse resultado e geramos UM exemplo ao vivo do "documento hipotetico".
import json
hb = json.load(open("_resultados/hyde_bench.json", encoding="utf-8"))
n = hb["n"]
print(f"hit-rate no benchmark completo (n={n}):   densa {hb['densa_hits'] / n:.0%}   |   HyDE {hb['hyde_hits'] / n:.0%}")
print(f"ambos acertam em {hb['both']} perguntas; so a densa em {hb['densa_only']}; so o HyDE em {hb['hyde_only']}")
print("-> recuperam trechos diferentes, mas empatam no acerto global (sem ganho liquido).")

# exemplo ao vivo do 'documento hipotetico' que o HyDE escreve (e que vira a consulta densa):
bench = json.load(open("benchmark/perguntas.json", encoding="utf-8"))["perguntas"]
ex = bench[0]
print("\nPergunta:", ex["pergunta"])
print("Documento hipotetico gerado:", rc.hyde_doc(ex["pergunta"])[:300])